In [5]:
import requests
from pymongo import MongoClient
import time

In [6]:
# Configuration
LASTFM_API_KEY = "a5529673c7042a37aa417b16d0b34b8e"
LASTFM_USER = "christo761"
MONGO_URI = "mongodb://localhost:27017/"
MONGO_DB_NAME = "MusicLibrary"
MONGO_COLLECTION_NAME = "user_plays"
PAGE_LIMIT = 200  # Max number of results per page (200 is Last.fm's limit)

In [7]:
# Connect to MongoDB
client = MongoClient(MONGO_URI)
db = client[MONGO_DB_NAME]
collection = db[MONGO_COLLECTION_NAME]

In [8]:
def fetch_lastfm_scrobbles(user, api_key, page=1, limit=PAGE_LIMIT):
    url = "https://ws.audioscrobbler.com/2.0/"
    params = {
        "method": "user.getrecenttracks",
        "user": user,
        "api_key": api_key,
        "format": "json",
        "limit": limit,
        "page": page
    }
    response = requests.get(url, params=params)
    if response.status_code == 200:
        return response.json()
    else:
        print(f"Failed to fetch data: {response.status_code} - {response.text}")
        return None

In [9]:
def save_to_mongo(data):
    if data:
        collection.insert_many(data)
        print(f"Inserted {len(data)} records into MongoDB.")

In [10]:
def main():
    print("Starting the script...")
    page = 1
    total_pages = 1  # Assume at least one page initially

    while page <= total_pages:
        print(f"Fetching page {page} of {total_pages}...")
        response = fetch_lastfm_scrobbles(LASTFM_USER, LASTFM_API_KEY, page)
        if response:
            tracks = response.get("recenttracks", {}).get("track", [])
            if tracks:
                # Transform data as needed before inserting into MongoDB
                transformed_data = [
                    {
                        "artist": track["artist"]["#text"],
                        "track": track["name"],
                        "album": track.get("album", {}).get("#text", ""),
                        "timestamp": track.get("date", {}).get("uts", None)
                    }
                    for track in tracks
                ]
                save_to_mongo(transformed_data)
            total_pages = int(response.get("recenttracks", {}).get("@attr", {}).get("totalPages", 1))
        else:
            print("Error fetching data. Exiting...")
            break
        page += 1
        time.sleep(1)  # To avoid hitting API rate limits

    print("Script completed.")

In [11]:
main()

Starting the script...
Fetching page 1 of 1...
Inserted 200 records into MongoDB.
Fetching page 2 of 288...
Inserted 200 records into MongoDB.
Fetching page 3 of 288...
Inserted 200 records into MongoDB.
Fetching page 4 of 288...
Inserted 200 records into MongoDB.
Fetching page 5 of 288...
Inserted 200 records into MongoDB.
Fetching page 6 of 288...
Inserted 200 records into MongoDB.
Fetching page 7 of 288...
Inserted 200 records into MongoDB.
Fetching page 8 of 288...
Inserted 200 records into MongoDB.
Fetching page 9 of 288...
Inserted 200 records into MongoDB.
Fetching page 10 of 288...
Inserted 200 records into MongoDB.
Fetching page 11 of 288...
Inserted 200 records into MongoDB.
Fetching page 12 of 288...
Inserted 200 records into MongoDB.
Fetching page 13 of 288...
Inserted 200 records into MongoDB.
Fetching page 14 of 288...
Inserted 200 records into MongoDB.
Fetching page 15 of 288...
Inserted 200 records into MongoDB.
Fetching page 16 of 288...
Inserted 200 records into Mongo

In [10]:
collection

Collection(Database(MongoClient(host=['localhost:27017'], document_class=dict, tz_aware=False, connect=True), 'MusicLibrary'), 'user_plays')

In [ ]:
query = {
    "$expr": {
        "$eq": ["$artist", "Anathema"]  # Find documents where discount_price < price
    }
}

pipeline = [
    {
        "$lookup": {
            "from": "AlbumNormalize",  # Target collection to join with
            "let": {  # Variables to use in the $expr condition
                "customer_id": "$customer_id",
                "order_date": "$order_date"
            },
            "pipeline": [
                {
                    "$match": {
                        "$expr": {
                            "$and": [
                                { "$eq": ["$customer_id", "$$customer_id"] },
                                { "$eq": ["$registration_date", "$$order_date"] }
                            ]
                        }
                    }
                }
            ],
            "as": "customer_info"  # Name of the output array field
        }
    },
    {
        "$project": {  # Optional: Control the fields in the output
            "_id": 1,
            "customer_id": 1,
            "order_date": 1,
            "customer_info.name": 1
        }
    }
]

for doc in collection.find(query):
    print(doc)

{'_id': ObjectId('674f79711bbba0b91d8193c1'), 'artist': 'Anathema', 'track': 'The Silent Enigma (Studio)', 'album': 'The Silent Enigma', 'timestamp': '1728996983'}
{'_id': ObjectId('674f79711bbba0b91d8193c2'), 'artist': 'Anathema', 'track': 'Cerulean Twilight (Studio)', 'album': 'The Silent Enigma', 'timestamp': '1728996557'}
{'_id': ObjectId('674f79711bbba0b91d8193c3'), 'artist': 'Anathema', 'track': 'Nocturnal Emission (Studio)', 'album': 'The Silent Enigma', 'timestamp': '1728996297'}
{'_id': ObjectId('674f79711bbba0b91d8193c4'), 'artist': 'Anathema', 'track': 'Sunset Of Age (Studio)', 'album': 'The Silent Enigma', 'timestamp': '1728995881'}
{'_id': ObjectId('674f79711bbba0b91d8193c5'), 'artist': 'Anathema', 'track': 'Alone (Studio)', 'album': 'The Silent Enigma', 'timestamp': '1728995617'}
{'_id': ObjectId('674f79711bbba0b91d8193c6'), 'artist': 'Anathema', 'track': 'Shroud Of Frost (Studio)', 'album': 'The Silent Enigma', 'timestamp': '1728995165'}
{'_id': ObjectId('674f79711bbba0b

In [21]:
user_plays_collection = db["user_plays"]
album_normalize_collection = db["AlbumNormalize"]

# Step 2: Define the aggregation pipeline
artist_filter = "Anathema"  # Replace with the artist you want to filter by

pipeline = [
    # Filter by the specified artist
    {
        "$match": {
            "artist": artist_filter
        }
    },
    # Perform a LEFT JOIN between user_plays and AlbumNormalize
    {
        "$lookup": {
            "from": "AlbumNormalize",  # Target collection for the join
            "let": {  # Define variables for the join condition
                "artist": "$artist",
                "album": "$album"
            },
            "pipeline": [
                {
                    "$match": {
                        "$expr": {
                            "$and": [
                                { "$eq": ["$Artist", "$$artist"] },
                                { "$eq": ["$Album", "$$album"] }
                            ]
                        }
                    }
                }
            ],
            "as": "normalized_data"
        }
    },
    # Add a field to use AlbumNormalized if available, otherwise fallback to album
    {
        "$addFields": {
            "final_album": {
                "$ifNull": [
                    { "$arrayElemAt": ["$normalized_data.AlbumNormalized", 0] },
                    "$album"
                ]
            }
        }
    },
    # Group by the final_album field
    {
        "$group": {
            "_id": "$final_album",
            "play_count": { "$sum": 1 }  # Count plays per group
        }
    },
    # Sort the result (optional)
    {
        "$sort": { "play_count": -1 }
    }
]

query = {
    "$expr": {
        "$eq": ["$artist", "Anathema"]  # Find documents where discount_price < price
    }
}

# Step 3: Execute the aggregation
result = user_plays_collection.aggregate(pipeline)

# Step 4: Print the results
print("Grouped Results:")
for doc in result:
    print(doc)


Grouped Results:
{'_id': 'The Silent Enigma', 'play_count': 1122}
{'_id': 'Judgement', 'play_count': 683}
{'_id': 'Eternity', 'play_count': 394}
{'_id': 'Alternative 4', 'play_count': 275}
{'_id': 'Weather Systems', 'play_count': 213}
{'_id': "We're Here Because We're Here", 'play_count': 210}
{'_id': 'A Natural Disaster', 'play_count': 197}
{'_id': 'Distant Satellites', 'play_count': 179}
{'_id': 'Falling Deeper', 'play_count': 166}
{'_id': 'Serenades', 'play_count': 158}
{'_id': 'A Fine Day to Exit', 'play_count': 156}
{'_id': 'Resonance', 'play_count': 141}
{'_id': 'Hindsight', 'play_count': 138}
{'_id': 'Universal', 'play_count': 136}
{'_id': 'The Optimist', 'play_count': 77}
{'_id': '', 'play_count': 77}
{'_id': 'A Sort of Homecoming', 'play_count': 75}
{'_id': 'Resonance 2', 'play_count': 71}
{'_id': 'Original Album Classics', 'play_count': 68}
{'_id': 'Pentecost III & The Crestfallen EP', 'play_count': 63}
{'_id': 'A Moment in Time', 'play_count': 28}
{'_id': 'All Faith Is Lost'

In [12]:
response2 = fetch_lastfm_scrobbles(LASTFM_USER, LASTFM_API_KEY, 1)

In [13]:
response2

{'recenttracks': {'track': [{'artist': {'mbid': '83d91898-7763-47d7-b03b-b92132375c47',
     '#text': 'Pink Floyd'},
    'streamable': '0',
    'image': [{'size': 'small',
      '#text': 'https://lastfm.freetls.fastly.net/i/u/34s/6af6a9a0d246464f976bef5193823322.png'},
     {'size': 'medium',
      '#text': 'https://lastfm.freetls.fastly.net/i/u/64s/6af6a9a0d246464f976bef5193823322.png'},
     {'size': 'large',
      '#text': 'https://lastfm.freetls.fastly.net/i/u/174s/6af6a9a0d246464f976bef5193823322.png'},
     {'size': 'extralarge',
      '#text': 'https://lastfm.freetls.fastly.net/i/u/300x300/6af6a9a0d246464f976bef5193823322.png'}],
    'mbid': '597eae0b-ccb6-4076-b997-0ce940586076',
    'album': {'mbid': '02a26364-1e0c-434b-bbba-c5d28fa91a4a',
     '#text': 'The Wall'},
    'name': 'Another Brick in the Wall, Pt. 2',
    'url': 'https://www.last.fm/music/Pink+Floyd/_/Another+Brick+in+the+Wall,+Pt.+2',
    'date': {'uts': '1775217442', '#text': '03 Apr 2026, 11:57'}},
   {'artist':